1. Import Library yang Dibutuhkan

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

2. Memuat dan Memproses Data (Preprocessing)

In [ ]:
# 1. Membaca data CSV
train_df = pd.read_csv('train_landmarks.csv')
val_df = pd.read_csv('val_landmarks.csv')

# 2. Memisahkan Fitur (X) dan Target/Label (Y)
X_train = train_df.drop('label', axis=1).values
y_train_labels = train_df['label'].values

X_val = val_df.drop('label', axis=1).values
y_val_labels = val_df['label'].values

# 3. Mengubah huruf (A-Z) menjadi angka (0-25) agar bisa dibaca komputer
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train_labels)
y_val_encoded = encoder.transform(y_val_labels)

# 4. Mengubah ke format One-Hot Encoding (wajib untuk klasifikasi lebih dari 2 kelas)
y_train = to_categorical(y_train_encoded)
y_val = to_categorical(y_val_encoded)

# 5. Reshape data X (CNN butuh input 3 Dimensi: jumlah_data, jumlah_fitur, channel)
# Kita punya 63 fitur (21 titik * 3 sumbu)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_val = np.reshape(X_val, (X_val.shape[0], X_val.shape[1], 1))

print(f"Bentuk data X_train: {X_train.shape}")
print(f"Bentuk data Y_train: {y_train.shape}")

3. Membangun Arsitektur CNN 1 Dimensi (1D)

In [ ]:
# Membuat model berurutan (Sequential)
model = Sequential([
    # Layer Ekstraksi Pola (Convolutional)
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    
    # Layer Perata (Flatten)
    Flatten(),
    
    # Layer Jaringan Syaraf Tiruan (Dense)
    Dense(128, activation='relu'),
    Dropout(0.5), # Mencegah model menghafal data (overfitting)
    
    # Layer Output (26 kelas untuk A-Z)
    Dense(26, activation='softmax')
])

# Mengatur optimizer dan cara menghitung error
model.compile(optimizer='adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

model.summary()

4. Melatih Model (Training)

In [ ]:
# Memulai proses training
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val)
)

5. Visualisasi Evaluasi Model

In [ ]:
# Membuat plot akurasi
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Grafik Akurasi Model')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.legend()

# Membuat plot loss (tingkat error)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Grafik Loss Model')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

6. Menyimpan Model dan Label

In [ ]:
# Menyimpan otak AI-nya
model.save('cnn_bisindo.h5')
print("Model berhasil disimpan sebagai 'cnn_bisindo.h5'!")

# Menyimpan kamus urutan kelas (agar A tetap dibaca A, bukan B)
np.save('classes_bisindo.npy', encoder.classes_)
print("Daftar label berhasil disimpan!")